In [1]:
import pandas as pd
from scipy.stats import poisson

columns = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "B365H", "B365D", "B365A"]

train = pd.read_csv("https://www.football-data.co.uk/mmz4281/2425/E0.csv")[columns]
test = pd.read_csv("https://www.football-data.co.uk/mmz4281/2526/E0.csv")[columns]

print("Train (2024/25):", len(train), "matches")
print("Test (2025/26):", len(test), "matches")

Train (2024/25): 380 matches
Test (2025/26): 380 matches


In [2]:
train_teams = set(train["HomeTeam"])
test_teams = set(test["HomeTeam"])

print("Promoted (in test, not in train):", test_teams - train_teams)
print("Relegated (in train, not in test):", train_teams - test_teams)

Promoted (in test, not in train): {'Burnley', 'Sunderland', 'Leeds'}
Relegated (in train, not in test): {'Leicester', 'Ipswich', 'Southampton'}


In [3]:
def compute_strengths(data):
    home_avg = data["FTHG"].mean()
    away_avg = data["FTAG"].mean()

    home_attack = data.groupby("HomeTeam")["FTHG"].mean() / home_avg
    home_defence = data.groupby("HomeTeam")["FTAG"].mean() / away_avg
    away_attack = data.groupby("AwayTeam")["FTAG"].mean() / away_avg
    away_defence = data.groupby("AwayTeam")["FTHG"].mean() / home_avg

    strengths = pd.DataFrame({
        "home_attack": home_attack,
        "home_defence": home_defence,
        "away_attack": away_attack,
        "away_defence": away_defence,
    })

    return strengths, home_avg, away_avg


strengths, home_avg, away_avg = compute_strengths(train)
print(strengths.round(2))
print("League home goals avg per match:", round(home_avg, 2), "| League away goals avg per match:", round(away_avg, 2))

                home_attack  home_defence  away_attack  away_defence
Arsenal                1.22          0.63         1.26          0.59
Aston Villa            1.18          0.74         0.89          1.08
Bournemouth            0.80          0.59         1.30          1.04
Brentford              1.39          1.30         0.96          0.77
Brighton               1.04          0.96         1.33          1.15
Chelsea                1.22          0.67         1.07          0.87
Crystal Palace         0.83          0.96         1.00          0.87
Everton                0.90          0.85         0.59          0.73
Fulham                 0.94          1.11         1.00          0.83
Ipswich                0.49          1.63         0.81          1.32
Leicester              0.52          1.26         0.67          1.60
Liverpool              1.46          0.59         1.63          0.87
Man City               1.50          0.85         1.07          0.73
Man United             0.80       

In [4]:
promoted = list(test_teams - train_teams)
relegated = list(train_teams - test_teams)

promoted_strength = strengths.loc[relegated].mean()
print("Strength given to promoted teams:")
print(promoted_strength.round(2))

for team in promoted:
    strengths.loc[team] = promoted_strength

print(len(strengths), "teams in the table")

Strength given to promoted teams:
home_attack     0.49
home_defence    1.54
away_attack     0.65
away_defence    1.43
dtype: float64
23 teams in the table


In [5]:
def expected_goals(home, away):
    home_xg = strengths.loc[home, "home_attack"] * strengths.loc[away, "away_defence"] * home_avg
    away_xg = strengths.loc[away, "away_attack"] * strengths.loc[home, "home_defence"] * away_avg
    return home_xg, away_xg


def predict(home, away):
    home_xg, away_xg = expected_goals(home, away)

    home_win = 0
    draw = 0
    away_win = 0

    for h in range(10):
        for a in range(10):
            p = poisson.pmf(h, home_xg) * poisson.pmf(a, away_xg)
            if h > a:
                home_win += p
            elif h == a:
                draw += p
            else:
                away_win += p

    return home_win, draw, away_win


hw, d, aw = predict("Sunderland", "West Ham")
print("Sunderland win:", round(hw * 100, 1), "%")
print("Draw:", round(d * 100, 1), "%")
print("West Ham win:", round(aw * 100, 1), "%")

Sunderland win: 13.7 %
Draw: 21.6 %
West Ham win: 64.8 %


In [6]:
model_home = []
model_draw = []
model_away = []

for i in range(len(test)):
    home = test["HomeTeam"].iloc[i]
    away = test["AwayTeam"].iloc[i]

    hw, d, aw = predict(home, away)

    model_home.append(hw)
    model_draw.append(d)
    model_away.append(aw)

test["model_H"] = model_home
test["model_D"] = model_draw
test["model_A"] = model_away

test["margin"] = 1 / test["B365H"] + 1 / test["B365D"] + 1 / test["B365A"]
test["book_H"] = (1 / test["B365H"]) / test["margin"]
test["book_D"] = (1 / test["B365D"]) / test["margin"]
test["book_A"] = (1 / test["B365A"]) / test["margin"]

print(test[["HomeTeam", "AwayTeam", "model_H", "book_H", "model_A", "book_A"]].head(10).round(3))

        HomeTeam        AwayTeam  model_H  book_H  model_A  book_A
0      Liverpool     Bournemouth    0.646   0.730    0.166   0.112
1    Aston Villa       Newcastle    0.511   0.413    0.247   0.321
2       Brighton          Fulham    0.359   0.498    0.382   0.238
3     Sunderland        West Ham    0.137   0.294    0.648   0.425
4      Tottenham         Burnley    0.678   0.688    0.154   0.112
5         Wolves        Man City    0.215   0.151    0.554   0.651
6        Chelsea  Crystal Palace    0.527   0.577    0.223   0.208
7  Nott'm Forest       Brentford    0.404   0.436    0.277   0.274
8     Man United         Arsenal    0.138   0.260    0.645   0.480
9          Leeds         Everton    0.154   0.398    0.555   0.315


In [7]:
test["edge_H"] = test["model_H"] - test["book_H"]

bets = test[test["edge_H"] > 0.05].copy()

bets["profit"] = -1.0
bets.loc[bets["FTR"] == "H", "profit"] = bets["B365H"] - 1

print("Number of bets:", len(bets))
print("Total profit (£1 per bet):", round(bets["profit"].sum(), 2))
print("ROI:", round(bets["profit"].sum() / len(bets) * 100, 1), "%")

Number of bets: 111
Total profit (£1 per bet): -14.18
ROI: -12.8 %


In [8]:
print("Before:", type(test["Date"].iloc[0]))

train["Date"] = pd.to_datetime(train["Date"], dayfirst=True)
test["Date"] = pd.to_datetime(test["Date"], dayfirst=True)

print("After:", type(test["Date"].iloc[0]))
print(test["Date"].iloc[0])

Before: <class 'str'>
After: <class 'pandas.Timestamp'>
2025-08-15 00:00:00


In [9]:
i = 200
match_date = test["Date"].iloc[i]
print("Match:", test["HomeTeam"].iloc[i], "vs", test["AwayTeam"].iloc[i], "on", match_date.date())

before = test[test["Date"] < match_date]
history = pd.concat([train, before])

print("2025/26 matches already played:", len(before))
print("Total matches the model can learn from:", len(history))

Match: West Ham vs Nott'm Forest on 2026-01-06
2025/26 matches already played: 200
Total matches the model can learn from: 580


In [10]:
hist_strengths, hist_home_avg, hist_away_avg = compute_strengths(history)

print(hist_strengths.loc[["Sunderland", "Leeds", "Burnley", "Man United"]].round(2))

            home_attack  home_defence  away_attack  away_defence
Sunderland         1.04          0.59         0.37          0.72
Leeds              1.17          0.96         0.59          1.30
Burnley            0.52          0.96         0.89          1.69
Man United         0.92          1.05         0.95          0.97


In [11]:
def strengths_before(match_date):
    before = test[test["Date"] < match_date]
    history = pd.concat([train, before])

    s, h_avg, a_avg = compute_strengths(history)

    for team in promoted:
        home_games = (history["HomeTeam"] == team).sum()
        away_games = (history["AwayTeam"] == team).sum()
        if home_games < 5 or away_games < 5:
            s.loc[team] = promoted_strength

    return s, h_avg, a_avg

In [12]:
s, h_avg, a_avg = strengths_before(test["Date"].iloc[5])
print("Match 5:")
print(s.loc[["Sunderland", "Leeds", "Burnley"]].round(2))

s, h_avg, a_avg = strengths_before(test["Date"].iloc[200])
print("Match 200:")
print(s.loc[["Sunderland", "Leeds", "Burnley"]].round(2))

Match 5:
            home_attack  home_defence  away_attack  away_defence
Sunderland         0.49          1.54         0.65          1.43
Leeds              0.49          1.54         0.65          1.43
Burnley            0.49          1.54         0.65          1.43
Match 200:
            home_attack  home_defence  away_attack  away_defence
Sunderland         1.04          0.59         0.37          0.72
Leeds              1.17          0.96         0.59          1.30
Burnley            0.52          0.96         0.89          1.69


In [13]:
def predict_with(home, away, s, h_avg, a_avg):
    home_xg = s.loc[home, "home_attack"] * s.loc[away, "away_defence"] * h_avg
    away_xg = s.loc[away, "away_attack"] * s.loc[home, "home_defence"] * a_avg

    home_win = 0
    draw = 0
    away_win = 0

    for h in range(10):
        for a in range(10):
            p = poisson.pmf(h, home_xg) * poisson.pmf(a, away_xg)
            if h > a:
                home_win += p
            elif h == a:
                draw += p
            else:
                away_win += p

    return home_win, draw, away_win

In [14]:
s, h_avg, a_avg = strengths_before(test["Date"].iloc[3])
hw, d, aw = predict_with("Sunderland", "West Ham", s, h_avg, a_avg)
print("Sunderland win:", round(hw * 100, 1), "%")

Sunderland win: 13.7 %


In [15]:
live_home = []
live_draw = []
live_away = []

for i in range(len(test)):
    home = test["HomeTeam"].iloc[i]
    away = test["AwayTeam"].iloc[i]
    date = test["Date"].iloc[i]

    s, h_avg, a_avg = strengths_before(date)
    hw, d, aw = predict_with(home, away, s, h_avg, a_avg)

    live_home.append(hw)
    live_draw.append(d)
    live_away.append(aw)

test["live_H"] = live_home
test["live_D"] = live_draw
test["live_A"] = live_away

print(test[["Date", "HomeTeam", "AwayTeam", "model_H", "live_H", "book_H"]].iloc[[3, 100, 200, 300]].round(3))

          Date    HomeTeam       AwayTeam  model_H  live_H  book_H
3   2025-08-16  Sunderland       West Ham    0.137   0.137   0.294
100 2025-11-08   Tottenham     Man United    0.432   0.438   0.359
200 2026-01-06    West Ham  Nott'm Forest    0.222   0.242   0.313
300 2026-03-16   Brentford         Wolves    0.564   0.637   0.610


/var/folders/56/t82_80m974df5kqxhh74dzsw0000gn/T/ipykernel_54062/1464398786.py:21: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(test[["Date", "HomeTeam", "AwayTeam", "model_H", "live_H", "book_H"]].iloc[[3, 100, 200, 300]].round(3))


In [16]:
test["edge_live"] = test["live_H"] - test["book_H"]

bets = test[test["edge_live"] > 0.05].copy()

bets["profit"] = -1.0
bets.loc[bets["FTR"] == "H", "profit"] = bets["B365H"] - 1

print("Number of bets:", len(bets))
print("Total profit (£1 per bet):", round(bets["profit"].sum(), 2))
print("ROI:", round(bets["profit"].sum() / len(bets) * 100, 1), "%")

Number of bets: 111
Total profit (£1 per bet): -11.32
ROI: -10.2 %


In [17]:
test["is_H"] = (test["FTR"] == "H").astype(int)
test["is_D"] = (test["FTR"] == "D").astype(int)
test["is_A"] = (test["FTR"] == "A").astype(int)

print(test[["HomeTeam", "AwayTeam", "FTR", "is_H", "is_D", "is_A"]].head())

      HomeTeam     AwayTeam FTR  is_H  is_D  is_A
0    Liverpool  Bournemouth   H     1     0     0
1  Aston Villa    Newcastle   D     0     1     0
2     Brighton       Fulham   D     0     1     0
3   Sunderland     West Ham   H     1     0     0
4    Tottenham      Burnley   H     1     0     0


In [18]:
def brier(p_home, p_draw, p_away):
    score = (p_home - test["is_H"])**2 + (p_draw - test["is_D"])**2 + (p_away - test["is_A"])**2
    return score.mean()

base = train["FTR"].value_counts(normalize=True)

print("Baseline:", round(brier(base["H"], base["D"], base["A"]), 4))
print("August model:", round(brier(test["model_H"], test["model_D"], test["model_A"]), 4))
print("Live model:", round(brier(test["live_H"], test["live_D"], test["live_A"]), 4))
print("Bookmaker:", round(brier(test["book_H"], test["book_D"], test["book_A"]), 4))

Baseline: 0.6568
August model: 0.6707
Live model: 0.638
Bookmaker: 0.6115


In [19]:
import numpy as np

def log_loss(p_home, p_draw, p_away):
    p_actual = p_home * test["is_H"] + p_draw * test["is_D"] + p_away * test["is_A"]
    return -np.log(p_actual).mean()

print("Baseline:", round(log_loss(base["H"], base["D"], base["A"]), 4))
print("August model:", round(log_loss(test["model_H"], test["model_D"], test["model_A"]), 4))
print("Live model:", round(log_loss(test["live_H"], test["live_D"], test["live_A"]), 4))
print("Bookmaker:", round(log_loss(test["book_H"], test["book_D"], test["book_A"]), 4))

Baseline: 1.0847
August model: 1.1106
Live model: 1.0595
Bookmaker: 1.0185


In [20]:
def strengths_before_w(match_date, w):
    before = test[test["Date"] < match_date]
    history = pd.concat([train] + [before] * w)

    s, h_avg, a_avg = compute_strengths(history)

    for team in promoted:
        home_games = (before["HomeTeam"] == team).sum()
        away_games = (before["AwayTeam"] == team).sum()
        if home_games < 5 or away_games < 5:
            s.loc[team] = promoted_strength

    return s, h_avg, a_avg

In [21]:
s1, _, _ = strengths_before(test["Date"].iloc[300])
s2, _, _ = strengths_before_w(test["Date"].iloc[300], 1)
print((s1 - s2).abs().max().max())

0.0


In [22]:
def run_model(w):
    H = []
    D = []
    A = []

    for i in range(len(test)):
        home = test["HomeTeam"].iloc[i]
        away = test["AwayTeam"].iloc[i]
        date = test["Date"].iloc[i]

        s, h_avg, a_avg = strengths_before_w(date, w)
        hw, d, aw = predict_with(home, away, s, h_avg, a_avg)

        H.append(hw)
        D.append(d)
        A.append(aw)

    H = np.array(H)
    D = np.array(D)
    A = np.array(A)

    return brier(H, D, A), log_loss(H, D, A)


for w in [1, 2, 3, 5, 10]:
    b, l = run_model(w)
    print("Weight", w, "| Brier:", round(b, 4), "| Log loss:", round(l, 4))

Weight 1 | Brier: 0.638 | Log loss: 1.0595
Weight 2 | Brier: 0.6367 | Log loss: 1.0576
Weight 3 | Brier: 0.6373 | Log loss: 1.0586
Weight 5 | Brier: 0.6395 | Log loss: 1.0624
Weight 10 | Brier: 0.6452 | Log loss: 1.0727


In [23]:
def evaluate(old, new, w):
    # 1. Promoted and relegated teams
    old_teams = set(old["HomeTeam"])
    new_teams = set(new["HomeTeam"])
    prom = list(new_teams - old_teams)
    rel = list(old_teams - new_teams)

    old_strengths, _, _ = compute_strengths(old)
    prom_strength = old_strengths.loc[rel].mean()

    # 2. Predict every match with updating strengths
    H = []
    D = []
    A = []

    for i in range(len(new)):
        date = new["Date"].iloc[i]
        before = new[new["Date"] < date]
        history = pd.concat([old] + [before] * w)

        s, h_avg, a_avg = compute_strengths(history)

        for team in prom:
            if (before["HomeTeam"] == team).sum() < 5 or (before["AwayTeam"] == team).sum() < 5:
                s.loc[team] = prom_strength

        hw, d, aw = predict_with(new["HomeTeam"].iloc[i], new["AwayTeam"].iloc[i], s, h_avg, a_avg)
        H.append(hw)
        D.append(d)
        A.append(aw)

    H = np.array(H)
    D = np.array(D)
    A = np.array(A)

    # 3. Grade the predictions
    is_H = (new["FTR"] == "H").astype(int)
    is_D = (new["FTR"] == "D").astype(int)
    is_A = (new["FTR"] == "A").astype(int)

    brier_score = ((H - is_H)**2 + (D - is_D)**2 + (A - is_A)**2).mean()
    p_actual = H * is_H + D * is_D + A * is_A
    log_loss_score = -np.log(p_actual).mean()

    return brier_score, log_loss_score

In [24]:
b, l = evaluate(train, test, 2)
print("Brier:", round(b, 4), "| Log loss:", round(l, 4))

Brier: 0.6367 | Log loss: 1.0576


In [25]:
s2324 = pd.read_csv("https://www.football-data.co.uk/mmz4281/2324/E0.csv")[columns]
s2324["Date"] = pd.to_datetime(s2324["Date"], dayfirst=True)

print("Validation: 2023/24 → predict 2024/25")
for w in [1, 2, 3, 5, 10]:
    b, l = evaluate(s2324, train, w)
    print("Weight", w, "| Brier:", round(b, 4), "| Log loss:", round(l, 4))

Validation: 2023/24 → predict 2024/25
Weight 1 | Brier: 0.5875 | Log loss: 0.9845
Weight 2 | Brier: 0.5871 | Log loss: 0.9845
Weight 3 | Brier: 0.588 | Log loss: 0.986
Weight 5 | Brier: 0.5901 | Log loss: 0.9896
Weight 10 | Brier: 0.5946 | Log loss: 0.9964
